# RAG Assesment

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community ragas

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.3.3
Uninstalling langchain-core-1.3.3:
  Successfully uninstalled langchain-core-1.3.3
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: ragas 0.1.9
Uninstalling ragas-0.1.9:
  Successfully uninstalled ragas-0.1.9


In [ ]:
!pip install -U \
ragas \
langchain \
langchain-core \
langchain-openai \
langchain-community \
datasets \
openai \
python-dotenv

In [ ]:
import os, openai
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [ ]:
# -----------------------------------
# LLM Judge
# -----------------------------------
evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0
    )
)

In [ ]:
# -----------------------------------
# Dataset
# -----------------------------------
data_samples = {
    "question": [
        "When was Attention Is All You Need published and by whom?",
        "When was Attention Is All You Need published and by whom?",
    ],

    "answer": [
        "Attention Is All You Need was published by Taylor Swift in 2017",
        "Attention Is All You Need was published by Taylor Swift in 2003",
    ],

    # contexts must be list[list[str]]
    "contexts": [
        ["Attention Is All You Need was published by Vaswani et al. in 2017."],
        ["Attention Is All You Need was published by Vaswani et al. in 2017."],
    ],

    # required for answer_relevancy + recall metrics in recent versions
    "ground_truth": [
        "Attention Is All You Need was published by Vaswani et al. in 2017.",
        "Attention Is All You Need was published by Vaswani et al. in 2017.",
    ],
}

dataset = Dataset.from_dict(data_samples)

In [ ]:
# -----------------------------------
# Evaluation
# -----------------------------------
result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=evaluator_llm,
    raise_exceptions=False,
)

# -----------------------------------
# Show scores
# -----------------------------------
df = result.to_pandas()

print("\n=== AVERAGE SCORES ===\n")
print(df.mean(numeric_only=True))

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]


=== AVERAGE SCORES ===

faithfulness         0.250000
answer_relevancy     0.969922
context_precision    1.000000
context_recall       1.000000
dtype: float64


In [ ]:
df

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,When was Attention Is All You Need published a...,[Attention Is All You Need was published by Va...,Attention Is All You Need was published by Tay...,Attention Is All You Need was published by Vas...,0.5,0.969922,1.0,1.0
1,When was Attention Is All You Need published a...,[Attention Is All You Need was published by Va...,Attention Is All You Need was published by Tay...,Attention Is All You Need was published by Vas...,0.0,0.969922,1.0,1.0
